# Underwater Image Enhancement — Training Pipeline
**Architecture:** UNet + ResNet34 | **Loss:** L1 + SSIM + Perceptual | **Output:** `best_model.pth`

**Data sources:**
- Kaggle: UIEB · RUIE · UFO-120 · Sea-thru · UIEC
- HuggingFace: LSUI (3000 pairs) · UIEB-HF
- GitHub: EUVP (~12000 pairs) · Water-Net · Shallow-UWNet

Run **Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → ... → Cell 14** in order.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations pytorch-msssim
!pip install -q kaggle huggingface_hub datasets
print('Done')

## Cell 2 — Kaggle Setup + Download (5 datasets)

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

DATA_ROOT = '/content/datasets'
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp'}
os.makedirs(DATA_ROOT, exist_ok=True)

def count_images(folder):
    if not os.path.exists(folder): return 0
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXTS)

# ── Kaggle auth ────────────────────────────────────────────────────────────
kaggle_cfg = Path('/root/.kaggle/kaggle.json')
if not kaggle_cfg.exists():
    print('Upload kaggle.json  (Kaggle → Settings → API → Create New Token)')
    uploaded = files.upload()
    kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
    kaggle_cfg.write_bytes(uploaded['kaggle.json'])
    kaggle_cfg.chmod(0o600)
else:
    print('kaggle.json already present')

# ── Download each Kaggle dataset into its own sub-folder ───────────────────
KAGGLE_SOURCES = [
    # (slug,                                         sub-folder name)
    ('larjeck/uieb-dataset-raw-and-reference',       'uieb'),
    ('saravanakumarjsk/ruie-real-world-underwater-image-enhancement', 'ruie'),
    ('iamsouravbanerjee/ufo-120-dataset',            'ufo120'),
    ('nixiepixel/sea-thru-images',                   'seathru'),
    ('sakshaymahna/underwater-image-enhancement',    'uiec'),
]

for slug, name in KAGGLE_SOURCES:
    dest = os.path.join(DATA_ROOT, name)
    if count_images(dest) > 10:
        print(f'SKIP  {name}  ({count_images(dest)} imgs already)')
        continue
    os.makedirs(dest, exist_ok=True)
    print(f'Downloading {slug} ...')
    ret = os.system(f'kaggle datasets download -d {slug} -p {dest} --unzip -q 2>&1')
    n = count_images(dest)
    print(f'  {"OK" if n > 0 else "FAIL"}  {name}: {n} images')

# ── Summary ────────────────────────────────────────────────────────────────
print('\nKaggle downloads:')
for slug, name in KAGGLE_SOURCES:
    n = count_images(os.path.join(DATA_ROOT, name))
    print(f'  {n:>6}  {name}')

## Cell 3 — HuggingFace + GitHub Downloads

In [ ]:
import os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

DATA_ROOT = '/content/datasets'
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp'}

def count_images(folder):
    if not os.path.exists(folder): return 0
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXTS)

# ══════════════════════════════════════════════════════════════════════════
#  HUGGINGFACE
# ══════════════════════════════════════════════════════════════════════════
print('=== HuggingFace ===')

# ── HF-1: LSUI (Large Scale Underwater Image, streaming, save 3000 pairs)
lsui_raw = os.path.join(DATA_ROOT, 'lsui', 'raw')
lsui_ref = os.path.join(DATA_ROOT, 'lsui', 'ref')
if count_images(lsui_raw) > 10:
    print(f'SKIP  lsui  ({count_images(lsui_raw)} imgs)')
else:
    os.makedirs(lsui_raw, exist_ok=True)
    os.makedirs(lsui_ref, exist_ok=True)
    print('Downloading LSUI from HuggingFace (streaming, up to 3000 pairs) ...')
    try:
        import datasets as hf
        saved = 0
        # Try primary LSUI repo
        for repo_id in ['gengzhi/LSUI', 'xhwnjnby/UIEB']:
            try:
                ds = hf.load_dataset(repo_id, split='train', streaming=True)
                for row in ds:
                    if saved >= 3000: break
                    try:
                        # column names differ per dataset
                        inp = row.get('input') or row.get('image') or row.get('degraded')
                        lbl = row.get('label') or row.get('gt')   or row.get('enhanced') or inp
                        inp.save(os.path.join(lsui_raw, f'{saved:05d}.png'))
                        lbl.save(os.path.join(lsui_ref, f'{saved:05d}.png'))
                        saved += 1
                    except Exception:
                        continue
                if saved > 0:
                    print(f'  OK  lsui: {saved} pairs  (from {repo_id})')
                    break
            except Exception as e:
                print(f'  {repo_id} failed: {e}')
    except Exception as e:
        print(f'  HF import failed: {e}')

# ── HF-2: snapshot download of a UW dataset repo
hf2_dir = os.path.join(DATA_ROOT, 'hf_uie')
if count_images(hf2_dir) > 10:
    print(f'SKIP  hf_uie  ({count_images(hf2_dir)} imgs)')
else:
    print('Snapshot download: ziwei-zhang/WaterEnhancement ...')
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id='ziwei-zhang/WaterEnhancement',
            repo_type='dataset',
            local_dir=hf2_dir,
            ignore_patterns=['*.md','*.txt','*.py','.git*']
        )
        print(f'  OK  hf_uie: {count_images(hf2_dir)} images')
    except Exception as e:
        print(f'  FAIL: {e}')

# ══════════════════════════════════════════════════════════════════════════
#  GITHUB
# ══════════════════════════════════════════════════════════════════════════
print('\n=== GitHub ===')

GITHUB_REPOS = [
    {
        'name':    'euvp',
        'git_url': 'https://github.com/xahidbuffon/EUVP-Dataset',
        'zip_url': 'https://github.com/xahidbuffon/EUVP-Dataset/archive/refs/heads/master.zip',
        'zip_dir': 'EUVP-Dataset-master',
    },
    {
        'name':    'waternet',
        'git_url': 'https://github.com/Li-Chongyi/Water-Net',
        'zip_url': 'https://github.com/Li-Chongyi/Water-Net/archive/refs/heads/master.zip',
        'zip_dir': 'Water-Net-master',
    },
    {
        'name':    'shallow_uwnet',
        'git_url': 'https://github.com/mkartik/Shallow-UWnet',
        'zip_url': 'https://github.com/mkartik/Shallow-UWnet/archive/refs/heads/main.zip',
        'zip_dir': 'Shallow-UWnet-main',
    },
]

for repo in GITHUB_REPOS:
    dest = os.path.join(DATA_ROOT, repo['name'])
    if count_images(dest) > 10:
        print(f'SKIP  {repo["name"]}  ({count_images(dest)} imgs)')
        continue
    # Try git clone
    if os.path.exists(dest): shutil.rmtree(dest)
    print(f'Cloning {repo["name"]} ...')
    r = subprocess.run(
        ['git', 'clone', '--depth=1', repo['git_url'], dest],
        capture_output=True, text=True
    )
    if r.returncode == 0 and count_images(dest) > 0:
        print(f'  OK  {repo["name"]}: {count_images(dest)} images')
        continue
    # Fallback: zip download
    print(f'  git failed, trying zip ...')
    zip_path = f'/tmp/{repo["name"]}.zip'
    try:
        urllib.request.urlretrieve(repo['zip_url'], zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_ROOT)
        extracted = os.path.join(DATA_ROOT, repo['zip_dir'])
        if os.path.exists(extracted):
            os.rename(extracted, dest)
        os.remove(zip_path)
        print(f'  OK  {repo["name"]}: {count_images(dest)} images')
    except Exception as e:
        print(f'  FAIL  {repo["name"]}: {e}')

# ── Final tally ────────────────────────────────────────────────────────────
print('\nAll image folders:')
total = 0
for root, dirs, files in os.walk(DATA_ROOT):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    n = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
    if n >= 10:
        print(f'  {n:>6}  {root}')
        total += n
print(f'  ──────\n  {total:>6}  TOTAL')

## Cell 4 — Scan Disk and Build DATASET_CONFIG

In [ ]:
import os
from pathlib import Path

DATA_ROOT = '/content/datasets'
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp'}

def count_images(folder):
    if not os.path.exists(folder): return 0
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXTS)

def list_image_dirs(base, min_imgs=5):
    out = []
    if not os.path.exists(base): return out
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        n = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
        if n >= min_imgs:
            out.append((root, n))
    return sorted(out, key=lambda x: -x[1])

def find_pair(base):
    """Find (raw_dir, ref_dir) inside a folder using common naming conventions."""
    PAIR_NAMES = [
        ('trainA',    'trainB'),
        ('input',     'GT'),
        ('input',     'label'),
        ('input',     'ref'),
        ('raw',       'reference'),
        ('raw',       'ref'),
        ('degraded',  'clean'),
        ('underwater','gt'),
        ('A',         'B'),
        ('data',      'gt'),
    ]
    for root, dirs, _ in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        rp = Path(root)
        for a, b in PAIR_NAMES:
            da, db = rp/a, rp/b
            if count_images(str(da)) > 5 and count_images(str(db)) > 5:
                return str(da), str(db)
    # fallback: two sibling dirs with similar counts
    for root, dirs, _ in os.walk(base):
        rp = Path(root)
        img_dirs = sorted(
            [(d, count_images(str(rp/d))) for d in dirs if count_images(str(rp/d)) >= 20],
            key=lambda x: -x[1]
        )
        if len(img_dirs) >= 2:
            d1, n1 = img_dirs[0]; d2, n2 = img_dirs[1]
            if abs(n1-n2)/max(n1,n2) < 0.25:
                return str(rp/d1), str(rp/d2)
    return None, None

# ── Print full scan ────────────────────────────────────────────────────────
all_dirs = list_image_dirs(DATA_ROOT, min_imgs=5)
print(f'All image folders:\n')
for p, n in all_dirs:
    print(f'  {n:>6}  {p}')

# ── Build DATASET_CONFIG ───────────────────────────────────────────────────
DATASET_CONFIG = {}

for folder in sorted(Path(DATA_ROOT).iterdir()):
    if not folder.is_dir() or folder.name.startswith('_'): continue
    raw, ref = find_pair(str(folder))
    if raw and ref:
        n = min(count_images(raw), count_images(ref))
        if n > 0:
            DATASET_CONFIG[folder.name] = {'raw': raw, 'ref': ref}
            print(f'  PAIRED  {folder.name:<25}  {n} pairs')
    else:
        print(f'  SKIP    {folder.name:<25}  (no pair found)')

# ── MANUAL OVERRIDE — uncomment and fill if auto-detect missed a dataset ──
# DATASET_CONFIG['my_dataset'] = {
#     'raw': '/content/datasets/PASTE_DEGRADED_FOLDER',
#     'ref': '/content/datasets/PASTE_REFERENCE_FOLDER'
# }

total = sum(min(count_images(v['raw']), count_images(v['ref']))
            for v in DATASET_CONFIG.values())
print(f'\nDatasets : {list(DATASET_CONFIG.keys())}')
print(f'Total    : {total} pairs')

if total == 0:
    print('\nNOTHING LOADED.')
    print('1. Check Cell 2 and Cell 3 output for errors')
    print('2. Use MANUAL OVERRIDE above with the actual folder paths printed in the scan')
    print('3. Or run Cell 4b below to generate synthetic fallback data')

## Cell 4b — Synthetic Fallback (only if total=0 above)

In [ ]:
# Run ONLY if Cell 4 shows total=0.
# Generates 800 synthetic paired images so training can proceed.
import os, cv2, numpy as np
from pathlib import Path

DATA_ROOT = '/content/datasets'

def count_images(folder):
    if not os.path.exists(folder): return 0
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in {'.jpg','.png'})

existing = sum(min(count_images(v['raw']), count_images(v['ref']))
               for v in DATASET_CONFIG.values()) if DATASET_CONFIG else 0

if existing > 0:
    print(f'Real data available ({existing} pairs) — synthetic skipped')
else:
    N   = 800
    syn_raw = os.path.join(DATA_ROOT, 'synthetic', 'raw')
    syn_ref = os.path.join(DATA_ROOT, 'synthetic', 'ref')
    os.makedirs(syn_raw, exist_ok=True)
    os.makedirs(syn_ref, exist_ok=True)
    print(f'Generating {N} synthetic pairs ...')
    for i in range(N):
        h = w = 256
        clean = np.random.randint(60, 220, (h, w, 3), dtype=np.uint8)
        clean[:,:,0] = (clean[:,:,0] * 0.6).astype(np.uint8)
        deg = clean.astype(np.float32)
        deg[:,:,0] *= 0.5; deg[:,:,2] *= 1.3
        haze = np.random.uniform(0.3, 0.6)
        deg  = deg*(1-haze) + 180*haze + np.random.normal(0,8,(h,w,3))
        deg  = np.clip(deg, 0, 255).astype(np.uint8)
        cv2.imwrite(os.path.join(syn_raw,f'{i:04d}.png'), cv2.cvtColor(deg,   cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(syn_ref,f'{i:04d}.png'), cv2.cvtColor(clean, cv2.COLOR_RGB2BGR))
    DATASET_CONFIG['synthetic'] = {'raw': syn_raw, 'ref': syn_ref}
    print(f'Done — {N} pairs. Re-run Cell 6 now.')

## Cell 5 — Dataset Class

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 256
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

TRAIN_TRANSFORM = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.4),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
], additional_targets={'image1': 'image'})

VAL_TRANSFORM = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
], additional_targets={'image1': 'image'})


class PairedDataset(Dataset):
    """Paired (degraded, clean) image dataset."""

    def __init__(self, raw_dir, ref_dir):
        raw_files = sorted([f for f in Path(raw_dir).iterdir() if f.suffix.lower() in IMG_EXTS])
        ref_files = sorted([f for f in Path(ref_dir).iterdir() if f.suffix.lower() in IMG_EXTS])
        ref_map   = {f.stem: str(f) for f in ref_files}
        self.pairs = [(str(r), ref_map[r.stem]) for r in raw_files if r.stem in ref_map]
        if not self.pairs:
            n = min(len(raw_files), len(ref_files))
            self.pairs = [(str(raw_files[i]), str(ref_files[i])) for i in range(n)]
        print(f'  {len(self.pairs)} pairs  <-  {Path(raw_dir).name}')

    def __len__(self): return len(self.pairs)
    def get_raw(self, idx): return self.pairs[idx]


class SplitDataset(Dataset):
    """Subset with its own independent transform."""

    def __init__(self, base_ds, indices, transform):
        self.base      = base_ds
        self.indices   = indices
        self.transform = transform

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        raw_path, ref_path = self.base.get_raw(self.indices[idx])
        raw = np.array(Image.open(raw_path).convert('RGB'))
        ref = np.array(Image.open(ref_path).convert('RGB'))
        out = self.transform(image=raw, image1=ref)
        return out['image'], out['image1']


print('Dataset classes ready')

## Cell 6 — Build DataLoaders

In [ ]:
import os, torch
from torch.utils.data import ConcatDataset, DataLoader

DATA_ROOT = '/content/datasets'

if not DATASET_CONFIG:
    print('DATASET_CONFIG empty — re-scanning ...')
    for p, n in list_image_dirs(DATA_ROOT, min_imgs=5):
        print(f'  {n:>6}  {p}')
    raise RuntimeError(
        'No datasets loaded.\n'
        'A) Re-run Cells 2–4 and check for download errors.\n'
        'B) Run Cell 4b to use synthetic data.')

all_train, all_val = [], []

for name, paths in DATASET_CONFIG.items():
    raw_dir, ref_dir = paths['raw'], paths['ref']
    if not os.path.exists(raw_dir) or not os.path.exists(ref_dir):
        print(f'SKIP {name} — folder missing')
        continue
    ds = PairedDataset(raw_dir, ref_dir)
    if len(ds) == 0:
        print(f'SKIP {name} — 0 pairs')
        continue
    total   = len(ds)
    n_val   = max(1, int(0.1 * total))
    n_train = total - n_val
    idx     = torch.randperm(total, generator=torch.Generator().manual_seed(42)).tolist()
    all_train.append(SplitDataset(ds, idx[:n_train], TRAIN_TRANSFORM))
    all_val.append(  SplitDataset(ds, idx[n_train:], VAL_TRANSFORM))
    print(f'OK  {name:<28}  train={n_train}  val={n_val}')

if not all_train:
    raise RuntimeError('No datasets loaded. Run Cell 4b then re-run this cell.')

train_dataset = ConcatDataset(all_train)
val_dataset   = ConcatDataset(all_val)

BATCH_SIZE  = 8
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True)

print(f'\nTrain : {len(train_dataset):>6} samples  ({len(train_loader)} batches)')
print(f'Val   : {len(val_dataset):>6} samples  ({len(val_loader)} batches)')

## Cell 7 — Model: UNet + ResNet34

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = smp.Unet(
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 3,
    activation      = 'tanh'
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')

## Cell 8 — Loss Functions

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from pytorch_msssim import ssim


class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features
        self.slice = nn.Sequential(*list(vgg.children())[:16]).eval()
        for p in self.parameters(): p.requires_grad = False

    def forward(self, pred, target):
        p = (pred   + 1.0) / 2.0
        t = (target + 1.0) / 2.0
        return F.l1_loss(self.slice(p), self.slice(t))


_perc = PerceptualLoss().to(DEVICE)


def combined_loss(pred, target):
    l1_v   = F.l1_loss(pred, target)
    perc_v = _perc(pred, target)
    p01    = (pred   + 1.0) / 2.0
    t01    = (target + 1.0) / 2.0
    ssim_v = 1.0 - ssim(p01, t01, data_range=1.0, size_average=True)
    return 0.5 * l1_v + 0.3 * ssim_v + 0.2 * perc_v


print('Loss: 50% L1 + 30% SSIM + 20% Perceptual ready')

## Cell 9 — Optimizer and Scheduler

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS    = 50
LR        = 2e-4
SAVE_PATH = '/content/best_model.pth'

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f'AdamW lr={LR} | CosineAnnealingLR T_max={EPOCHS} | Epochs={EPOCHS}')

## Cell 10 — Training Loop

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from pytorch_msssim import ssim as compute_ssim


def train_one_epoch(model, loader, optimizer):
    model.train()
    running = 0.0
    for raw, ref in loader:
        raw = raw.to(DEVICE, non_blocking=True)
        ref = ref.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = combined_loss(model(raw), ref)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item()
    return running / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    tl = ts = tp = 0.0
    for raw, ref in loader:
        raw  = raw.to(DEVICE)
        ref  = ref.to(DEVICE)
        pred = model(raw)
        tl  += combined_loss(pred, ref).item()
        p01  = (pred + 1.0) / 2.0
        r01  = (ref  + 1.0) / 2.0
        ts  += compute_ssim(p01, r01, data_range=1.0, size_average=True).item()
        tp  += 10.0 * np.log10(1.0 / (F.mse_loss(p01, r01).item() + 1e-8))
    n = len(loader)
    return tl / n, ts / n, tp / n


best_val  = float('inf')
history   = {'train': [], 'val': [], 'ssim': [], 'psnr': []}

print(f"{'Ep':>4}  {'Train':>9}  {'Val':>9}  {'SSIM':>7}  {'PSNR':>7}  {'LR':>9}  Time")
print('-' * 62)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = train_one_epoch(model, train_loader, optimizer)
    vl, vs, vp = validate(model, val_loader)
    scheduler.step()

    history['train'].append(tr)
    history['val'].append(vl)
    history['ssim'].append(vs)
    history['psnr'].append(vp)

    lr  = optimizer.param_groups[0]['lr']
    sec = time.time() - t0
    tag = ''

    if vl < best_val:
        best_val = vl
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_loss': float(vl), 'val_ssim': float(vs),
                    'val_psnr': float(vp)}, SAVE_PATH)
        tag = '  <- saved'

    print(f'{epoch:>4}  {tr:>9.4f}  {vl:>9.4f}  {vs:>7.4f}  {vp:>7.2f}  {lr:>9.2e}  {sec:>3.0f}s{tag}')

print(f'\nDone. Best val_loss = {best_val:.4f}')

## Cell 11 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train'], label='Train'); axes[0].plot(history['val'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(history['ssim'], color='green');  axes[1].set_title('Val SSIM')
axes[2].plot(history['psnr'], color='orange'); axes[2].set_title('Val PSNR (dB)')
for ax in axes[1:]: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## Cell 12 — Visual Check: Degraded vs Enhanced vs Reference

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt

ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}  '
      f'SSIM={ckpt["val_ssim"]:.4f}  PSNR={ckpt["val_psnr"]:.2f} dB')

def denorm(t):
    arr = t.cpu().clamp(-1, 1).permute(1, 2, 0).numpy()
    return ((arr + 1.0) / 2.0 * 255).astype(np.uint8)

raw_b, ref_b = next(iter(val_loader))
with torch.no_grad():
    pred_b = model(raw_b.to(DEVICE)).cpu()

n = min(4, raw_b.shape[0])
fig, ax = plt.subplots(3, n, figsize=(4*n, 12))
for row, (label, batch) in enumerate(zip(
        ['Degraded', 'Enhanced', 'Reference'], [raw_b, pred_b, ref_b])):
    for col in range(n):
        ax[row, col].imshow(denorm(batch[col]))
        ax[row, col].axis('off')
        if col == 0: ax[row, col].set_ylabel(label, fontsize=11)
plt.tight_layout()
plt.savefig('/content/visual_check.png', dpi=150)
plt.show()
print('Saved visual_check.png')

## Cell 13 — Inference Function (bridge to formula pipeline)

In [ ]:
import cv2, torch, numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

_INFER_TF = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])


def enhance_image(img, model, device=DEVICE):
    """
    img   : file path (str) OR numpy uint8 HxWx3 RGB
    returns numpy uint8 HxWx3 RGB at original resolution
    """
    if isinstance(img, str):
        img = np.array(Image.open(img).convert('RGB'))
    orig_h, orig_w = img.shape[:2]
    t   = _INFER_TF(image=img)['image'].unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out = model(t)[0]
    arr = out.cpu().clamp(-1, 1).permute(1, 2, 0).numpy()
    sq  = ((arr + 1.0) / 2.0 * 255).astype(np.uint8)
    return cv2.resize(sq, (orig_w, orig_h), interpolation=cv2.INTER_LANCZOS4)


dummy  = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
result = enhance_image(dummy, model)
assert result.shape == (480, 640, 3)
print(f'enhance_image() OK:  {dummy.shape} -> {result.shape}')

## Cell 14 — Download Outputs

In [ ]:
import os
from google.colab import files

for path in [SAVE_PATH, '/content/training_curves.png', '/content/visual_check.png']:
    if os.path.exists(path):
        print(f'Downloading {path} ...')
        files.download(path)
    else:
        print(f'NOT FOUND: {path}')
print('Done')